# 16 — Upsert, Bearer Access & GraphQL Config

## What's new in v0.31.0

This notebook covers the five features added in the first PyPI release
targeting SurrealDB 3.0:

1. **`QuerySet.upsert()`** — Insert or update on conflict with `ON DUPLICATE KEY UPDATE`
2. **`QuerySet.bulk_upsert()`** — Batch upsert with shared conflict handling
3. **`DefineBearerAccess`** — Machine-to-machine API keys via `TYPE BEARER`
4. **`DefineGraphQLConfig`** — Configure SurrealDB's built-in GraphQL endpoint
5. **`RebuildIndex`** — Trigger index rebuilds after bulk imports

> **Requires:** SurrealDB 3.0+ and `surreal-orm >= 0.31.0`

## Setup

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from surreal_orm import BaseSurrealModel, SurrealConfigDict, SurrealDBConnectionManager, SurrealFunc

SurrealDBConnectionManager.set_connection(
    os.getenv("SURREALDB_URL", "ws://localhost:8000"),
    os.getenv("SURREALDB_USER", "root"),
    os.getenv("SURREALDB_PASS", "root"),
    os.getenv("SURREALDB_NAMESPACE", "ns"),
    os.getenv("SURREALDB_DATABASE", "db"),
)
print("Connection configured.")

Connection configured.


---

## 1. QuerySet.upsert() — Insert or Update on Conflict

The `upsert()` method generates an `INSERT INTO ... ON DUPLICATE KEY UPDATE`
statement when `on_conflict` is provided, or a plain `UPSERT ... SET` otherwise.

This is useful for idempotent writes — the same call creates the record if it
doesn't exist, or updates specific fields if it does.

### Key features

- **`defaults`** — Fields for the new record
- **`id`** — Optional record ID (auto-generated if omitted)
- **`on_conflict`** — Fields to update when the record already exists
- **`SurrealFunc`** support — Use server-side expressions like `login_count += 1`
- **Proper ID escaping** — IDs starting with digits are safely backtick-escaped

### Important

`ON DUPLICATE KEY UPDATE` expressions run in the context of the **existing**
record. On the first insert there is no existing record, so expressions like
`login_count + 1` would fail (`NONE + 1`). Use a plain `upsert()` (without
`on_conflict`) to create the record first, then use `on_conflict` for
subsequent upserts. Compound operators like `+=` work the same way.

In [2]:
class Player(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="players")

    id: str | None = None
    name: str = ""
    login_count: int = 0
    score: int = 0


# Create the table
await BaseSurrealModel.raw_query(
    "DEFINE TABLE players SCHEMALESS;"
)
print("Table created.")

Table created.


In [3]:
# --- Step 1: Create a record with plain upsert (no on_conflict) ---
player = await Player.objects().upsert(
    defaults={"name": "Alice", "login_count": 1, "score": 100},
    id="players:alice",
)
print(f"Created: {player.name}, logins={player.login_count}, score={player.score}")


# --- Step 2: Upsert again with ON DUPLICATE KEY UPDATE ---
# Alice already exists, so only the on_conflict expression runs.
# login_count += 1 increments the existing value.
player = await Player.objects().upsert(
    defaults={"name": "Alice", "login_count": 1, "score": 100},
    id="players:alice",
    on_conflict={"login_count": SurrealFunc("login_count += 1")},
)
print(f"After upsert: {player.name}, logins={player.login_count}, score={player.score}")
# login_count is now 2, score unchanged

Created: Alice, logins=1, score=100
After upsert: Alice, logins=2, score=100


In [4]:
# --- SurrealFunc with compound assignment operators ---
# First, create Bob with a plain upsert
player = await Player.objects().upsert(
    defaults={"name": "Bob", "score": 50},
    id="players:bob",
)
print(f"Bob created: score={player.score}")

# Upsert again with += — score increments by 10
player = await Player.objects().upsert(
    defaults={"name": "Bob", "score": 50},
    id="players:bob",
    on_conflict={"score": SurrealFunc("score += 10")},
)
print(f"Bob updated: score={player.score}")  # 60

Bob created: score=50
Bob updated: score=60


In [5]:
# --- Plain upsert (no on_conflict) ---
# Overwrites the entire record if it exists
player = await Player.objects().upsert(
    defaults={"name": "Charlie", "login_count": 1, "score": 200},
    id="players:charlie",
)
print(f"Plain upsert: {player.name}, score={player.score}")

# --- Upsert without ID (auto-generated) ---
player = await Player.objects().upsert(
    defaults={"name": "Random Player", "score": 0},
)
print(f"Auto-ID: {player.get_id()}, name={player.name}")

Plain upsert: Charlie, score=200
Auto-ID: nl70tre6w025uy2d4wjl, name=Random Player


---

## 2. QuerySet.bulk_upsert() — Batch Upsert

`bulk_upsert()` processes multiple model instances in a single query batch.
A shared `on_conflict` dict is applied to all instances.

Use `atomic=True` to wrap the entire batch in a transaction.

In [6]:
# Step 1: Create a batch of players (plain upsert, no on_conflict)
batch = [
    Player(id="players:p1", name="Player 1", score=10, login_count=1),
    Player(id="players:p2", name="Player 2", score=20, login_count=1),
    Player(id="players:p3", name="Player 3", score=30, login_count=1),
]

results = await Player.objects().bulk_upsert(batch)

print("Initial batch:")
for p in results:
    print(f"  {p.name}: score={p.score}, logins={p.login_count}")

print()

# Step 2: Run again WITH on_conflict — login_count increments
results = await Player.objects().bulk_upsert(
    batch,
    on_conflict={"login_count": SurrealFunc("login_count += 1")},
)

print("After second bulk_upsert:")
for p in results:
    print(f"  {p.name}: logins={p.login_count}")

Initial batch:
  Player 1: score=10, logins=1
  Player 2: score=20, logins=1
  Player 3: score=30, logins=1

After second bulk_upsert:
  Player 1: logins=2
  Player 2: logins=2
  Player 3: logins=2


In [7]:
# --- Atomic bulk_upsert (wrapped in a transaction) ---
atomic_batch = [
    Player(id="players:t1", name="TX Player 1", score=100),
    Player(id="players:t2", name="TX Player 2", score=200),
]

results = await Player.objects().bulk_upsert(atomic_batch, atomic=True)
print("Atomic upsert (all-or-nothing):")
for p in results:
    print(f"  {p.name}: score={p.score}")

Atomic upsert (all-or-nothing):


In [8]:
# Clean up players
await BaseSurrealModel.raw_query("REMOVE TABLE players;")
print("Players table removed.")

Players table removed.


---

## 3. Bearer Access — Machine-to-Machine API Keys

SurrealDB 3.0 adds `DEFINE ACCESS ... TYPE BEARER` for issuing API keys
that don't require user credentials. This is ideal for:

- Service-to-service authentication
- CI/CD pipelines
- Background workers

### Migration operation

Use `DefineBearerAccess` in your migration files to create the access definition.
Then use `grant_bearer_key()` and `revoke_bearer_key()` at runtime.

In [9]:
from surreal_orm.migrations.operations import DefineBearerAccess

# Generate the DEFINE ACCESS statement
bearer = DefineBearerAccess(
    name="api_keys",
    bearer_for="USER",
    duration_grant="90d",
    duration_session="24h",
)

print("Generated SQL:")
print(bearer.forwards())
print()
print(f"Rollback: {bearer.backwards()}")
print(f"Description: {bearer.describe()}")

Generated SQL:
DEFINE ACCESS api_keys ON DATABASE TYPE BEARER FOR USER
    DURATION FOR GRANT 90d, FOR SESSION 24h;

Rollback: REMOVE ACCESS api_keys ON DATABASE;
Description: Define bearer access api_keys


In [10]:
# Apply the access definition
await BaseSurrealModel.raw_query(bearer.forwards())
print("Bearer access 'api_keys' defined.")

# To grant a key, a database-level user must exist first.
# The GRANT syntax requires a specific user identifier:
#   ACCESS api_keys ON DATABASE GRANT FOR USER <username>
#
# Example (requires a DB user):
#   result = await BaseSurrealModel.raw_query(
#       "ACCESS api_keys ON DATABASE GRANT FOR USER my_service"
#   )
#   print(result)  # [{"id": "...", "key": "surreal-bearer-..."}]

print("GRANT syntax: ACCESS api_keys ON DATABASE GRANT FOR USER <username>")
print("REVOKE syntax: ACCESS api_keys ON DATABASE REVOKE <grant_id>")

Bearer access 'api_keys' defined.
GRANT syntax: ACCESS api_keys ON DATABASE GRANT FOR USER <username>
REVOKE syntax: ACCESS api_keys ON DATABASE REVOKE <grant_id>


In [11]:
# Clean up bearer access
await BaseSurrealModel.raw_query("REMOVE ACCESS api_keys ON DATABASE;")
print("Bearer access removed.")

Bearer access removed.


### ORM-level bearer key management

If your model uses `AuthenticatedUserMixin`, you can grant and revoke keys
directly from the model class:

```python
from surreal_orm.auth import AuthenticatedUserMixin

class ServiceAccount(AuthenticatedUserMixin, BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="service_accounts",
        access_name="api_keys",
    )
    id: str | None = None
    name: str

# Grant a bearer key
key_info = await ServiceAccount.grant_bearer_key(
    user_id="service_accounts:worker1",
)
# Returns: {"id": "access_grant:...", "key": "surreal-bearer-..."}

# Revoke a bearer key
await ServiceAccount.revoke_bearer_key(key_info["id"])
```

Both methods validate their inputs to prevent SurrealQL injection.

---

## 4. GraphQL Config — Configure the Built-In GraphQL Endpoint

SurrealDB 3.0 includes a built-in GraphQL endpoint. Use `DefineGraphQLConfig`
and `RemoveGraphQLConfig` migration operations to configure which tables and
functions are exposed.

### Modes

| Mode      | Behavior                                       |
|-----------|-------------------------------------------------|
| `AUTO`    | Expose all tables/functions automatically       |
| `NONE`    | Expose nothing (disabled)                       |
| `INCLUDE` | Expose only the listed tables/functions         |
| `EXCLUDE` | Expose everything except the listed ones        |

In [12]:
from surreal_orm.migrations.operations import DefineGraphQLConfig, RemoveGraphQLConfig

# Expose all tables, no functions
config_auto = DefineGraphQLConfig(
    tables_mode="AUTO",
    functions_mode="NONE",
)
print("Auto mode:")
print(config_auto.forwards())
print()

# Expose only specific tables
config_include = DefineGraphQLConfig(
    tables_mode="INCLUDE",
    tables_list=["users", "products"],
    functions_mode="AUTO",
)
print("Include mode:")
print(config_include.forwards())
print()

# Expose everything except internal tables
config_exclude = DefineGraphQLConfig(
    tables_mode="EXCLUDE",
    tables_list=["_migrations", "_audit_log"],
    functions_mode="AUTO",
)
print("Exclude mode:")
print(config_exclude.forwards())

Auto mode:
DEFINE CONFIG GRAPHQL TABLES AUTO FUNCTIONS NONE;

Include mode:
DEFINE CONFIG GRAPHQL TABLES INCLUDE users, products FUNCTIONS AUTO;

Exclude mode:
DEFINE CONFIG GRAPHQL TABLES EXCLUDE _migrations, _audit_log FUNCTIONS AUTO;


In [13]:
# RemoveGraphQLConfig disables GraphQL by setting TABLES NONE FUNCTIONS NONE
remove_gql = RemoveGraphQLConfig()
print(f"Disable GraphQL: {remove_gql.forwards()}")
print(f"Description: {remove_gql.describe()}")

Disable GraphQL: DEFINE CONFIG GRAPHQL TABLES NONE FUNCTIONS NONE;
Description: Remove GraphQL config


### Schema diffing

The migration system tracks GraphQL config via `GraphQLConfigState`.
When you run `makemigrations`, changes to the config are detected
automatically and generate the appropriate `DefineGraphQLConfig` or
`RemoveGraphQLConfig` operations.

In [14]:
from surreal_orm.migrations.state import GraphQLConfigState

# Represent a GraphQL config state
state = GraphQLConfigState(
    tables_mode="INCLUDE",
    tables_list=["users", "products"],
    functions_mode="NONE",
    functions_list=[],
)

print(f"Tables: {state.tables_mode} {state.tables_list}")
print(f"Functions: {state.functions_mode} {state.functions_list}")

Tables: INCLUDE ['users', 'products']
Functions: NONE []


---

## 5. RebuildIndex — Reindex After Bulk Imports

After loading large amounts of data, indexes may need to be rebuilt.
The `RebuildIndex` migration operation generates `REBUILD INDEX name ON table;`.

This operation is **irreversible** — there is no undo for a rebuild.

In [15]:
from surreal_orm.migrations.operations import RebuildIndex

# Basic rebuild
rebuild = RebuildIndex(table="products", name="idx_name")
print(f"SQL: {rebuild.forwards()}")
print(f"Description: {rebuild.describe()}")
print()

# With IF EXISTS (safe for migrations that may run on fresh databases)
rebuild_safe = RebuildIndex(table="products", name="idx_name", if_exists=True)
print(f"Safe SQL: {rebuild_safe.forwards()}")
print()

# Irreversible — backwards() raises
try:
    rebuild.backwards()
except Exception as e:
    print(f"Rollback error: {e.__class__.__name__}: {e}")

SQL: REBUILD INDEX idx_name ON products;
Description: Rebuild index idx_name on products

Safe SQL: REBUILD INDEX IF EXISTS idx_name ON products;



### Using RebuildIndex in a migration

```python
# migrations/0003_rebuild_after_import.py
from surreal_orm.migrations.operations import RebuildIndex

operations = [
    RebuildIndex(table="documents", name="ft_content", if_exists=True),
    RebuildIndex(table="documents", name="vec_embedding", if_exists=True),
]
```

Run after a bulk data import to ensure full-text and vector indexes are up to date.

---

## Summary

| Feature | ORM API | SurrealDB Syntax |
|---------|---------|------------------|
| Upsert | `QuerySet.upsert(defaults, id, on_conflict)` | `INSERT INTO ... ON DUPLICATE KEY UPDATE` |
| Bulk upsert | `QuerySet.bulk_upsert(instances, on_conflict, atomic)` | Batched `INSERT INTO` |
| Bearer access | `DefineBearerAccess(name, bearer_for)` | `DEFINE ACCESS ... TYPE BEARER` |
| Bearer keys | `Model.grant_bearer_key()` / `.revoke_bearer_key()` | `ACCESS name GRANT` / `REVOKE` |
| GraphQL config | `DefineGraphQLConfig(tables_mode, functions_mode)` | `DEFINE CONFIG GRAPHQL TABLES ... FUNCTIONS ...` |
| Rebuild index | `RebuildIndex(table, name, if_exists)` | `REBUILD INDEX name ON table` |

### Key points

- **`upsert()`** uses `INSERT INTO` with `ON DUPLICATE KEY UPDATE` — not `UPSERT SET`
- **`SurrealFunc`** expressions work in both `defaults` and `on_conflict` dicts
- **Compound operators** (`+=`, `-=`) are output as-is without `field =` wrapping
- **Bearer keys** are validated with `validate_identifier()` to prevent injection
- **`RebuildIndex`** is irreversible — use `if_exists=True` for safety
- All features require **SurrealDB 3.0+**